# ETF_V8 V2 JQ-like Robustness 实验

目标：以 `ETF_V2 周频机器学习轮动实验` 为主线候选，先补齐训推一致和稳健性拆解，而不是继续堆复杂模型。

本 notebook 做四件事：

1. 复用 V2 的宽 ETF 池、趋势/R² 特征、`ret5d/alpha5d` label 和 LGB 回归训练逻辑。
2. 导出 V2 风格 pkl，特别覆盖 `train20210101_20251231 + ret5d` 这个当前候选主模型。
3. 新增 JQ-like 离线执行模拟：周一/下一交易日开盘近似成交、100 股最小单位、手续费、滑点、现金约束、买不起不补候选等。
4. 输出 top1/top3、不同训练窗、不同 label、不同资金规模、是否主题 cap 的年度收益和汇总表。

重要说明：

- 默认主线是 `v2_raw_no_refill`，尽量贴近你发来的 V2 回测文件：不限制主题、不用正交池、topN 直接取、买不起就跳过不补。
- 主题去重、refill、等权重只作为诊断对照，不作为默认策略。
- 如果缺少 JoinQuant API，但已有 CSV 缓存，本 notebook 可完成训练/普通 proxy；JQ-like open 价格模拟会跳过并提示。

In [ ]:
# =========================
# 0. Config
# =========================
try:
    from jqdata import *
except Exception:
    # 本地语法检查/离线读 CSV 时允许没有 jqdata；重建数据和 open 价模拟仍需要在聚宽环境运行。
    pass

import os
import gc
import math
import pickle
import datetime
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OUT_DIR = "etf_ml_v8_v2_jq_like_robustness_outputs"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# V8 输出。
PANEL_CSV = os.path.join(OUT_DIR, "etf_ml_v8_weekly_panel.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v8_score_panel.csv")
MODEL_MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v8_model_manifest.csv")
CLOSE_PROXY_CSV = os.path.join(OUT_DIR, "etf_ml_v8_close_proxy_weekly.csv")
OPEN_PRICE_CSV = os.path.join(OUT_DIR, "etf_ml_v8_rebalance_open_prices.csv")
JQ_LIKE_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v8_jq_like_targets.csv")
JQ_LIKE_EQUITY_CSV = os.path.join(OUT_DIR, "etf_ml_v8_jq_like_equity.csv")
JQ_LIKE_SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v8_jq_like_summary.csv")
JQ_LIKE_YEARLY_CSV = os.path.join(OUT_DIR, "etf_ml_v8_jq_like_yearly.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v8_latest_targets.csv")
MODEL_DIR = os.path.join(OUT_DIR, "models")
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# 可选读取旧 V2 panel，避免重复重建。没有则自动重建。
V2_PANEL_CANDIDATES = [
    "etf_ml_v2_outputs/etf_ml_v2_weekly_panel.csv",
    "../etf_ml_v2_outputs/etf_ml_v2_weekly_panel.csv",
    "quant-research/机器学习策略/notebooks/etf_ml_v2_outputs/etf_ml_v2_weekly_panel.csv",
]
USE_EXISTING_V2_PANEL_IF_FOUND = True
FORCE_REBUILD_DATA = False
REBUILD_DATA = None  # None 表示自动：找不到缓存才重建。
FORCE_RECOMPUTE_REBALANCE_DATE = True  # JQ-like 必须用 feature_date 的下一交易日，不能信任旧缓存里的 rebalance_date。
FULL_OPEN_PRICE_COVERAGE = True  # 抓取 score_df 全量 code/date 的 open 价，避免持仓不在 topN 时无法估值。

START_DATE = "2016-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
LABEL_HORIZON_DAYS = 5
MIN_LISTING_DAYS = 180
MIN_AVG_MONEY_20 = 20000000.0
ETF_CHUNK_SIZE = 120
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42

# V2 特征：趋势/R2 + 横截面 rank + 池内趋势广度。
TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1)
POOL_CONTEXT_WINDOW = 25

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
]

# 主题分组只用于诊断对照；主线不启用。
ETF_GROUP_RULES = [
    ("bank", ["银行"]),
    ("innovative_drug", ["创新药", "新药", "医药", "医疗", "生物", "药"]),
    ("oil_gas", ["油气", "石油", "能源"]),
    ("coal", ["煤炭"]),
    ("semiconductor", ["半导", "芯片", "集成电路"]),
    ("software_ai", ["软件", "人工智能", "AI", "云计算", "大数据", "计算机", "信创"]),
    ("broker", ["证券", "券商"]),
    ("military", ["军工", "国防", "航空航天", "通用航空"]),
    ("gold", ["黄金"]),
    ("nonferrous", ["有色", "稀有金属", "稀土"]),
    ("consumer", ["消费", "食品", "酒"]),
    ("real_estate", ["地产", "房地产"]),
    ("new_energy", ["新能源", "光伏", "电池", "锂电", "储能"]),
]
ETF_NAME_CACHE = {}

# 重点包含你当前关心的 2021-2025 训练窗。
TRAIN_WINDOWS = [
    ("train20210101_20231231", "2021-01-01", "2023-12-31"),
    ("train20190101_20241231", "2019-01-01", "2024-12-31"),
    ("train20210101_20251231", "2021-01-01", "2025-12-31"),
]
TARGET_SPECS = [
    ("ret5d", "future_ret_5d"),
    ("alpha5d", "target_alpha_5d"),
]

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60",
    "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60",
    "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for _w in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % _w,
        "trend_r2_%s" % _w,
        "trend_score_%s" % _w,
        "trend_vol_%s" % _w,
        "trend_score_vol_adj_%s" % _w,
        "trend_simple_ann_%s" % _w,
    ])
CONTEXT_FEATURE_COLS = [
    "pool_breadth_%s" % POOL_CONTEXT_WINDOW,
    "pool_median_vol_%s" % POOL_CONTEXT_WINDOW,
]
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + c for c in RAW_FEATURE_COLS]
FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + CONTEXT_FEATURE_COLS

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.90,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "min_gain_to_split": 0.0,
    "verbose": -1,
    "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 180

# JQ-like 模拟参数。
RUN_JQ_LIKE_SIM = True
INITIAL_CASH_LIST = [200000.0, 500000.0, 1000000.0]
STOCK_NUM_LIST = [1, 3]
SELECT_POLICIES = ["v2_raw_no_refill", "raw_refill_next", "theme_cap1_refill"]
MAIN_SELECT_POLICY = "v2_raw_no_refill"
TOP_CANDIDATE_BUFFER = 20
LOT_SIZE = 100
FIXED_SLIPPAGE = 0.001
COMMISSION_RATE = 0.0001

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
print("out dir:", OUT_DIR)
print("features:", len(FEATURE_COLS))
print("train windows:", TRAIN_WINDOWS)
print("targets:", TARGET_SPECS)

In [ ]:
# =========================
# 1. Data helpers: V2 feature and label rebuild
# =========================
def first_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


def to_timestamp(x):
    return pd.Timestamp(x).normalize()


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_weekly_feature_dates(start_date, end_date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("Need JoinQuant API to rebuild weekly feature dates; provide PANEL_CSV or run in JoinQuant.")
    if len(days) == 0:
        return []
    s = pd.Series(days)
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%W")):
        out.append(pd.Timestamp(gdf.max()).normalize())
    return out


def get_next_trade_date(date):
    try:
        days = pd.to_datetime(get_trade_days(start_date=date, count=2))
    except NameError:
        raise RuntimeError("Need JoinQuant API to derive rebalance dates.")
    if len(days) < 2:
        return pd.NaT
    return pd.Timestamp(days[1]).normalize()


def should_exclude_name(name):
    text = str(name)
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and kw in text:
            return True
    return False


def classify_etf_group(code, name):
    text = str(name)
    for group_name, keywords in ETF_GROUP_RULES:
        for kw in keywords:
            if kw and kw in text:
                return group_name
    return "single_" + str(code)


def get_etf_universe_on_date(date):
    try:
        sec_df = get_all_securities(["etf"], date=date)
    except NameError:
        raise RuntimeError("Need JoinQuant API to rebuild ETF universe; provide cached panel or run in JoinQuant.")
    except Exception as err:
        print("get_all_securities failed", date, err)
        return []
    if sec_df is None or sec_df.empty:
        return []
    out = []
    name_map = {}
    for code, row in sec_df.iterrows():
        try:
            start_date = row.get("start_date", None)
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
            feature_dt = pd.Timestamp(date).date()
            if feature_dt - start_date < datetime.timedelta(days=MIN_LISTING_DAYS):
                continue
            name = row.get("display_name", "")
            if should_exclude_name(name):
                continue
            out.append(code)
            name_map[code] = str(name)
        except Exception:
            continue
    ETF_NAME_CACHE[str(pd.Timestamp(date).date())] = name_map
    return out


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def calc_trend_metrics(close, days):
    if len(close) <= days:
        return {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    try:
        slope, intercept = np.polyfit(x, y, 1, w=weights)
    except Exception:
        return {"ann": np.nan, "r2": np.nan, "score": np.nan, "vol": np.nan, "score_vol_adj": np.nan, "simple_ann": np.nan}
    ann = math.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * np.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {"ann": ann, "r2": r2, "score": score, "vol": vol, "score_vol_adj": score_vol_adj, "simple_ann": simple_ann}


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    for col in ["open", "high", "low", "close", "volume", "money"]:
        if col not in df.columns:
            return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None
    ret = close.pct_change()
    rec = {"code": code}
    rec["ret_1"] = safe_ret(close, 1)
    rec["ret_5"] = safe_ret(close, 5)
    rec["ret_10"] = safe_ret(close, 10)
    rec["ret_20"] = safe_ret(close, 20)
    rec["ret_60"] = safe_ret(close, 60)
    rec["vol_5"] = ret.tail(5).std()
    rec["vol_20"] = ret.tail(20).std()
    rec["vol_60"] = ret.tail(60).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1.0
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1.0
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1.0
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1.0
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1.0
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1.0
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1.0
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1.0
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1.0
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1.0
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for w in TREND_WINDOWS:
        tm = calc_trend_metrics(close, w)
        rec["trend_ann_%s" % w] = tm["ann"]
        rec["trend_r2_%s" % w] = tm["r2"]
        rec["trend_score_%s" % w] = tm["score"]
        rec["trend_vol_%s" % w] = tm["vol"]
        rec["trend_score_vol_adj_%s" % w] = tm["score_vol_adj"]
        rec["trend_simple_ann_%s" % w] = tm["simple_ann"]
    return rec


def fetch_feature_rows(feature_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            price_df = get_price(etf_chunk, end_date=feature_date, frequency="daily", fields=fields, count=PRICE_HISTORY_COUNT, panel=False, fq="pre", skip_paused=False)
        except Exception as err:
            print("price feature chunk failed", feature_date, err)
            continue
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    if len(rows) == 0:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    return out.replace([np.inf, -np.inf], np.nan)


def fetch_future_returns(feature_date, etfs):
    try:
        td = pd.to_datetime(get_trade_days(start_date=feature_date, count=LABEL_HORIZON_DAYS + 1))
    except Exception as err:
        print("future trade days failed", feature_date, err)
        return pd.DataFrame(columns=["code", "future_ret_5d", "next_date"])
    if len(td) < LABEL_HORIZON_DAYS + 1:
        return pd.DataFrame(columns=["code", "future_ret_5d", "next_date"])
    next_date = pd.Timestamp(td[-1]).normalize()
    rows = []
    for etf_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        try:
            px = get_price(etf_chunk, start_date=feature_date, end_date=next_date, frequency="daily", fields=["close"], panel=False, fq="pre", skip_paused=False)
        except Exception as err:
            print("future price chunk failed", feature_date, err)
            continue
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time")
            if len(one) < 2:
                continue
            start_close = float(one.iloc[0]["close"])
            end_close = float(one.iloc[-1]["close"])
            if start_close <= 0:
                continue
            rows.append({"code": code, "future_ret_5d": end_close / start_close - 1.0, "next_date": next_date})
    return pd.DataFrame(rows)


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        if col in out.columns:
            out["rank_" + col] = out[col].rank(pct=True)
    return out


def add_pool_context_features(df):
    out = df.copy()
    ann_col = "trend_ann_%s" % POOL_CONTEXT_WINDOW
    vol_col = "trend_vol_%s" % POOL_CONTEXT_WINDOW
    if ann_col in out.columns and len(out) > 0:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = float((out[ann_col] > 0).mean())
    else:
        out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = np.nan
    if vol_col in out.columns and len(out) > 0:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = float(out[vol_col].median())
    else:
        out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = np.nan
    return out


def build_one_feature_date(feature_date):
    etfs = get_etf_universe_on_date(feature_date)
    if len(etfs) == 0:
        return pd.DataFrame()
    feature_df = fetch_feature_rows(feature_date, etfs)
    if feature_df.empty:
        return pd.DataFrame()
    if "money_mean_20" in feature_df.columns:
        feature_df = feature_df[feature_df["money_mean_20"] >= MIN_AVG_MONEY_20].copy()
    if feature_df.empty:
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    label_df = fetch_future_returns(feature_date, list(feature_df["code"]))
    if label_df.empty:
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    if out.empty:
        return out
    name_map = ETF_NAME_CACHE.get(str(pd.Timestamp(feature_date).date()), {})
    out["name"] = out["code"].map(name_map).fillna("")
    out["etf_group"] = out.apply(lambda r: classify_etf_group(r["code"], r.get("name", "")), axis=1)
    out = add_rank_features(out)
    out["feature_date"] = pd.Timestamp(feature_date).normalize()
    out["rebalance_date"] = out["feature_date"].apply(get_next_trade_date)
    out["target_alpha_5d"] = out["future_ret_5d"] - out["future_ret_5d"].median()
    return out


def build_weekly_panel():
    feature_dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("feature weeks:", len(feature_dates), "from", feature_dates[0] if feature_dates else None, "to", feature_dates[-1] if feature_dates else None)
    parts = []
    for dt in tqdm(feature_dates, desc="build etf weekly panel"):
        one = build_one_feature_date(dt)
        if one is not None and not one.empty:
            parts.append(one)
        if len(parts) > 0 and len(parts) % 20 == 0:
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("No ETF weekly samples were built. Check ETF universe/data access.")
    panel = pd.concat(parts, ignore_index=True, sort=False)
    label_quality = panel.groupby("feature_date")["future_ret_5d"].agg(["count", lambda s: (s == 0).mean()])
    label_quality.columns = ["sample_count", "zero_return_rate"]
    bad_dates = set(label_quality[label_quality["zero_return_rate"] >= 0.98].index)
    if bad_dates:
        print("drop incomplete label dates:", sorted([str(pd.Timestamp(x).date()) for x in bad_dates]))
        panel = panel[~panel["feature_date"].isin(bad_dates)].copy()
    for c in ["feature_date", "rebalance_date", "next_date"]:
        if c in panel.columns:
            panel[c] = pd.to_datetime(panel[c])
    return panel

In [ ]:
# =========================
# 2. Build or load V2-style panel
# =========================
def normalize_panel_dates(df):
    out = df.copy()
    for c in ["feature_date", "rebalance_date", "next_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c])
    # JQ-like 评估必须模拟真实回测：feature_date 生成信号，下一交易日 9:35 调仓。
    # 旧 V2 缓存可能把 rebalance_date 写成 feature_date，这会把周五收盘信号误当周五交易。
    if FORCE_RECOMPUTE_REBALANCE_DATE or "rebalance_date" not in out.columns:
        try:
            unique_dates = sorted(pd.to_datetime(out["feature_date"].dropna().unique()))
            next_map = {}
            for dt in tqdm(unique_dates, desc="derive rebalance dates"):
                next_map[pd.Timestamp(dt).normalize()] = get_next_trade_date(dt)
            out["rebalance_date"] = out["feature_date"].map(next_map)
        except Exception as err:
            print("rebalance_date unavailable until JoinQuant API is available:", err)
            if "rebalance_date" not in out.columns:
                out["rebalance_date"] = pd.NaT
    if "etf_group" not in out.columns:
        if "name" in out.columns:
            out["etf_group"] = out.apply(lambda r: classify_etf_group(r.get("code", ""), r.get("name", "")), axis=1)
        else:
            out["etf_group"] = out["code"].astype(str).map(lambda x: "single_" + x)
    return out

source_panel = None
if USE_EXISTING_V2_PANEL_IF_FOUND:
    source_panel = first_existing_path(V2_PANEL_CANDIDATES)

if REBUILD_DATA is None:
    do_rebuild = bool(FORCE_REBUILD_DATA or ((not os.path.exists(PANEL_CSV)) and source_panel is None))
else:
    do_rebuild = bool(REBUILD_DATA)

if do_rebuild:
    panel_df = build_weekly_panel()
    panel_df.to_csv(PANEL_CSV, index=False)
elif os.path.exists(PANEL_CSV):
    panel_df = pd.read_csv(PANEL_CSV)
    print("loaded V8 panel cache:", PANEL_CSV)
elif source_panel is not None:
    panel_df = pd.read_csv(source_panel)
    print("loaded existing V2 panel:", source_panel)
    panel_df.to_csv(PANEL_CSV, index=False)
else:
    raise IOError("No panel cache found and rebuild is disabled/unavailable.")

panel_df = normalize_panel_dates(panel_df)
print("panel shape:", panel_df.shape)
print(panel_df[["feature_date", "rebalance_date", "next_date"]].agg(["min", "max"]))
print("sample per week:")
print(panel_df.groupby("feature_date")["code"].count().describe())
display(panel_df.head())

In [ ]:
# =========================
# 3. Train V2-style models, score OOS, export pkl bundles
# =========================
META_COLS = ["code", "name", "etf_group", "feature_date", "rebalance_date", "next_date", "future_ret_5d", "target_alpha_5d"]
SCORE_EXPORT_COLS = META_COLS + ["score", "model_name", "model_file", "target_name", "target_col", "train_tag", "train_start", "train_end"]
PREDICT_CHUNK_SIZE = 50000


def clean_train_df(df, target_col):
    cols = [c for c in META_COLS + FEATURE_COLS if c in df.columns]
    out = df.loc[:, cols].replace([np.inf, -np.inf], np.nan)
    out = out[~out[target_col].isnull()].copy()
    return out


def append_csv(df, path):
    if df is None or df.empty:
        return
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)


def train_lgb_or_fallback(train_df, target_col, feature_cols):
    X_raw = train_df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    fill_values = X_raw.median().to_dict()
    X = X_raw.fillna(pd.Series(fill_values)).fillna(0).astype(np.float32)
    y = train_df[target_col].astype(np.float32).values
    try:
        import lightgbm as lgb
        dtrain = lgb.Dataset(X[feature_cols], label=y, feature_name=list(feature_cols), free_raw_data=True)
        model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
        backend = "lightgbm"
        del dtrain
    except Exception as err:
        print("LightGBM unavailable or failed, fallback to sklearn RandomForestRegressor:", err)
        from sklearn.ensemble import RandomForestRegressor
        model = RandomForestRegressor(n_estimators=260, max_depth=6, min_samples_leaf=30, random_state=RANDOM_SEED, n_jobs=1)
        model.fit(X[feature_cols], y)
        backend = "sklearn_random_forest"
    del X_raw, X, y
    gc.collect()
    return model, fill_values, backend


def predict_model_chunked(model, fill_values, df, feature_cols, chunk_size=PREDICT_CHUNK_SIZE):
    out = np.empty(len(df), dtype=np.float32)
    if len(df) == 0:
        return out
    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))
        part = df.iloc[start:end]
        X_raw = part.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
        X = X_raw.fillna(pd.Series(fill_values)).fillna(0).astype(np.float32)
        pred = np.asarray(model.predict(X[feature_cols])).reshape(-1).astype(np.float32)
        out[start:end] = pred
        del X_raw, X, pred, part
        gc.collect()
    return out


def make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end):
    return {
        "objective": "etf_ml_weekly_lgb_v2",
        "model": model,
        "model_backend": backend,
        "target_name": target_name,
        "target_col": target_col,
        "feature_cols": list(FEATURE_COLS),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS),
        "context_feature_cols": list(CONTEXT_FEATURE_COLS),
        "trend_windows": list(TREND_WINDOWS),
        "trend_weight_end": TREND_WEIGHT_END,
        "price_history_count": PRICE_HISTORY_COUNT,
        "pool_context_window": POOL_CONTEXT_WINDOW,
        "fill_values": dict(fill_values),
        "lookback_days": LOOKBACK_DAYS,
        "label_horizon_days": LABEL_HORIZON_DAYS,
        "min_listing_days": MIN_LISTING_DAYS,
        "min_avg_money_20": MIN_AVG_MONEY_20,
        "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
        "stock_num": 3,
        "benchmark": BENCHMARK,
        "train_tag": train_tag,
        "train_start": train_start,
        "train_end": train_end,
        "research_version": "etf_ml_v8_v2_jq_like_robustness",
        "created_note": "V8 keeps V2 model logic and adds JQ-like execution diagnostics; no theme cap in default backtest.",
    }


# Streaming mode: do not keep feature-bearing score tables for every model in memory.
manifest_rows = []
panel_fit = panel_df.copy()

for _p in [SCORE_CSV, MODEL_MANIFEST_CSV]:
    if os.path.exists(_p):
        os.remove(_p)

for train_tag, train_start, train_end in tqdm(TRAIN_WINDOWS, desc="train windows"):
    train_start_ts = pd.Timestamp(train_start)
    train_end_ts = pd.Timestamp(train_end)
    train_mask = (panel_fit["feature_date"] >= train_start_ts) & (panel_fit["next_date"] <= train_end_ts)
    score_mask = panel_fit["next_date"] > train_end_ts
    if (not bool(train_mask.any())) or (not bool(score_mask.any())):
        print("skip window", train_tag, "train", int(train_mask.sum()), "score", int(score_mask.sum()))
        continue

    score_cols = [c for c in META_COLS + FEATURE_COLS if c in panel_fit.columns]
    score_base = panel_fit.loc[score_mask, score_cols].copy()

    for target_name, target_col in tqdm(TARGET_SPECS, desc=train_tag, leave=False):
        if target_col not in panel_fit.columns:
            print("skip target missing", target_col)
            continue
        train_df = clean_train_df(panel_fit.loc[train_mask], target_col)
        if train_df.empty:
            print("skip empty train", train_tag, target_name)
            continue
        print("training", train_tag, target_name, "samples", len(train_df), "weeks", train_df["feature_date"].nunique())

        model, fill_values, backend = train_lgb_or_fallback(train_df, target_col, FEATURE_COLS)
        bundle = make_bundle(model, fill_values, backend, target_name, target_col, train_tag, train_start, train_end)
        model_name = "ml_v2_%s_%s" % (target_name, train_tag)
        model_file = "model_etf_ml_v8_v2_lgb_%s_%s.pkl" % (target_name, train_tag)
        model_path = os.path.join(MODEL_DIR, model_file)
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)

        scored = score_base.reindex(columns=[c for c in META_COLS if c in score_base.columns]).copy()
        scored["score"] = predict_model_chunked(model, fill_values, score_base, FEATURE_COLS)
        scored["model_name"] = model_name
        scored["model_file"] = model_file
        scored["target_name"] = target_name
        scored["target_col"] = target_col
        scored["train_tag"] = train_tag
        scored["train_start"] = train_start
        scored["train_end"] = train_end
        append_csv(scored.reindex(columns=[c for c in SCORE_EXPORT_COLS if c in scored.columns]), SCORE_CSV)

        manifest_rows.append({
            "model_name": model_name,
            "model_file": model_file,
            "target_name": target_name,
            "target_col": target_col,
            "train_tag": train_tag,
            "train_start": train_start,
            "train_end": train_end,
            "backend": backend,
            "train_rows": int(len(train_df)),
            "train_weeks": int(train_df["feature_date"].nunique()),
            "score_rows": int(len(scored)),
            "score_weeks": int(scored["feature_date"].nunique()),
            "feature_count": int(len(FEATURE_COLS)),
        })

        del scored, model, bundle, train_df, fill_values
        gc.collect()

    del score_base
    gc.collect()

manifest_df = pd.DataFrame(manifest_rows)
if not manifest_df.empty:
    manifest_df.to_csv(MODEL_MANIFEST_CSV, index=False)

if os.path.exists(SCORE_CSV):
    score_df = pd.read_csv(SCORE_CSV)
    for _c in ["feature_date", "rebalance_date", "next_date"]:
        if _c in score_df.columns:
            score_df[_c] = pd.to_datetime(score_df[_c])
else:
    score_df = pd.DataFrame()

# The remaining cells only need compact score_df, not the feature-bearing panel copy.
del panel_fit
try:
    gc.collect()
except Exception:
    pass

print("score_df:", score_df.shape)
display(manifest_df)

In [ ]:
# =========================
# 4. Close-to-close proxy: fast but not final model selection metric
# =========================
def summarize_returns(ret_series):
    r = pd.Series(ret_series).dropna().astype(float)
    if len(r) == 0:
        return {"periods": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "sharpe": np.nan}
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    std = r.std()
    return {
        "periods": int(len(r)),
        "cum_ret": float(nav.iloc[-1] - 1.0),
        "mean_ret": float(r.mean()),
        "win_rate": float((r > 0).mean()),
        "max_drawdown": float(dd.min()),
        "sharpe": float(r.mean() / std) if (not pd.isnull(std)) and std > 0 else np.nan,
    }


def select_rows_by_policy(gdf, stock_num, policy):
    sorted_df = gdf.sort_values("score", ascending=False).copy()
    if policy == "theme_cap1_refill":
        selected = []
        groups = {}
        for _, row in sorted_df.iterrows():
            grp = row.get("etf_group", classify_etf_group(row.get("code", ""), row.get("name", "")))
            if groups.get(grp, 0) >= 1:
                continue
            selected.append(row)
            groups[grp] = groups.get(grp, 0) + 1
            if len(selected) >= stock_num:
                break
        return pd.DataFrame(selected)
    return sorted_df.head(stock_num)


def build_close_proxy(score_data):
    rows = []
    if score_data is None or score_data.empty:
        return pd.DataFrame()
    for (model_name, dt), gdf in tqdm(score_data.groupby(["model_name", "feature_date"]), desc="close proxy"):
        for stock_num in STOCK_NUM_LIST:
            for policy in ["v2_raw_no_refill", "theme_cap1_refill"]:
                top = select_rows_by_policy(gdf, stock_num, policy)
                if top.empty:
                    continue
                rows.append({
                    "model_name": model_name,
                    "model_file": top["model_file"].iloc[0],
                    "target_name": top["target_name"].iloc[0],
                    "train_tag": top["train_tag"].iloc[0],
                    "stock_num": int(stock_num),
                    "select_policy": policy,
                    "feature_date": dt,
                    "next_date": top["next_date"].iloc[0],
                    "target_count": int(len(top)),
                    "targets": ",".join(top["code"].astype(str).tolist()),
                    "target_groups": ",".join(top.get("etf_group", pd.Series([""] * len(top))).astype(str).tolist()),
                    "ret_5d": float(top["future_ret_5d"].mean()),
                    "median_ret_5d": float(gdf["future_ret_5d"].median()),
                    "excess_vs_median_5d": float(top["future_ret_5d"].mean() - gdf["future_ret_5d"].median()),
                })
    return pd.DataFrame(rows)

close_proxy_df = build_close_proxy(score_df)
close_summary_rows = []
if not close_proxy_df.empty:
    for keys, gdf in close_proxy_df.groupby(["model_name", "stock_num", "select_policy"]):
        model_name, stock_num, policy = keys
        row = {"model_name": model_name, "stock_num": stock_num, "select_policy": policy}
        row.update({"proxy_" + k: v for k, v in summarize_returns(gdf["ret_5d"]).items()})
        row.update(gdf.iloc[0][["model_file", "target_name", "train_tag"]].to_dict())
        close_summary_rows.append(row)
close_summary_df = pd.DataFrame(close_summary_rows)
if not close_summary_df.empty:
    close_summary_df = close_summary_df.sort_values(["proxy_cum_ret", "proxy_max_drawdown"], ascending=[False, False])
close_proxy_df.to_csv(CLOSE_PROXY_CSV, index=False)
display(close_summary_df.head(30))

In [ ]:
# =========================
# 5. Fetch rebalance open prices for JQ-like simulator
# =========================
def build_candidate_table_for_prices(score_data, topn):
    if score_data is None or score_data.empty:
        return pd.DataFrame(columns=["feature_date", "rebalance_date", "code"])
    if FULL_OPEN_PRICE_COVERAGE:
        # Full score-universe coverage is still not enough: a bought ETF may fall out of
        # a later week's score panel but must still be valued/sold like JoinQuant can do.
        # Therefore, for each model we request every ever-scored ETF on every rebalance date
        # in that model's evaluation span. This is intentionally a little over-complete.
        base = score_data.reindex(columns=["model_name", "rebalance_date", "code"]).dropna().copy()
        base["rebalance_date"] = pd.to_datetime(base["rebalance_date"])
        rows = []
        for model_name, gdf in tqdm(base.groupby("model_name"), desc="price coverage models"):
            dates = sorted(pd.to_datetime(gdf["rebalance_date"].dropna().unique()))
            codes = sorted(gdf["code"].astype(str).dropna().unique().tolist())
            for dt in dates:
                for code in codes:
                    rows.append({"model_name": model_name, "rebalance_date": pd.Timestamp(dt).normalize(), "code": code})
        return pd.DataFrame(rows).drop_duplicates()
    rows = []
    for (model_name, dt), gdf in tqdm(score_data.groupby(["model_name", "feature_date"]), desc="candidate table"):
        gdf = gdf.sort_values("score", ascending=False).head(topn).copy()
        for rank_idx, (_, row) in enumerate(gdf.iterrows(), 1):
            rows.append({
                "model_name": model_name,
                "feature_date": row.get("feature_date"),
                "rebalance_date": row.get("rebalance_date"),
                "code": row.get("code"),
                "rank": int(rank_idx),
            })
    return pd.DataFrame(rows)


def fetch_open_prices_for_pairs(pair_df):
    if pair_df is None or pair_df.empty:
        return pd.DataFrame(columns=["date", "code", "open_price"])
    pairs = pair_df[["rebalance_date", "code"]].dropna().drop_duplicates().copy()
    pairs["rebalance_date"] = pd.to_datetime(pairs["rebalance_date"])
    rows = []
    for dt, gdf in tqdm(pairs.groupby("rebalance_date"), desc="fetch open prices"):
        codes = sorted(gdf["code"].astype(str).unique().tolist())
        for code_chunk in chunks(codes, ETF_CHUNK_SIZE):
            try:
                px = get_price(code_chunk, end_date=dt, frequency="daily", fields=["open"], count=1, panel=False, fq="pre", skip_paused=False)
            except NameError:
                raise RuntimeError("Need JoinQuant API to fetch rebalance open prices.")
            except Exception as err:
                print("open price fetch failed", dt, err)
                continue
            if px is None or px.empty:
                continue
            for _, row in px.iterrows():
                code = row.get("code")
                op = row.get("open")
                if pd.isnull(op) or float(op) <= 0:
                    continue
                rows.append({"date": pd.Timestamp(dt).normalize(), "code": str(code), "open_price": float(op)})
    return pd.DataFrame(rows)

if RUN_JQ_LIKE_SIM:
    candidate_df = build_candidate_table_for_prices(score_df, TOP_CANDIDATE_BUFFER)
    need_refetch = True
    if os.path.exists(OPEN_PRICE_CSV):
        cached = pd.read_csv(OPEN_PRICE_CSV)
        cached["date"] = pd.to_datetime(cached["date"])
        # Reuse cache only when it covers almost all required rebalance-date/code pairs.
        req = candidate_df[["rebalance_date", "code"]].dropna().drop_duplicates().copy()
        req["rebalance_date"] = pd.to_datetime(req["rebalance_date"])
        chk = req.merge(cached.rename(columns={"date": "rebalance_date"}), on=["rebalance_date", "code"], how="left")
        miss = int(chk["open_price"].isnull().sum()) if len(chk) else 0
        if miss == 0:
            open_price_df = cached
            need_refetch = False
            print("loaded complete open price cache:", OPEN_PRICE_CSV, open_price_df.shape)
        else:
            print("open price cache incomplete, refetch required pairs. missing=", miss, "required=", len(chk))
    if need_refetch:
        try:
            open_price_df = fetch_open_prices_for_pairs(candidate_df)
            open_price_df.to_csv(OPEN_PRICE_CSV, index=False)
        except Exception as err:
            print("JQ-like open price fetch skipped:", err)
            open_price_df = pd.DataFrame(columns=["date", "code", "open_price"])
else:
    open_price_df = pd.DataFrame(columns=["date", "code", "open_price"])

print("open prices:", open_price_df.shape)
display(open_price_df.head())

In [ ]:
# =========================
# 6. JQ-like weekly execution simulator
# =========================
def make_price_map(open_df):
    mp = {}
    if open_df is None or open_df.empty:
        return mp
    tmp = open_df.copy()
    tmp["date"] = pd.to_datetime(tmp["date"])
    for _, row in tmp.iterrows():
        mp[(pd.Timestamp(row["date"]).normalize(), str(row["code"]))] = float(row["open_price"])
    return mp


def get_open_price(price_map, date, code):
    return price_map.get((pd.Timestamp(date).normalize(), str(code)), np.nan)


def can_buy_lot(price, target_value):
    if pd.isnull(price) or price <= 0:
        return False
    buy_price = float(price) + FIXED_SLIPPAGE
    return target_value * 0.98 >= buy_price * LOT_SIZE


def select_target_codes(gdf, stock_num, policy, target_value, price_map):
    sorted_df = gdf.sort_values("score", ascending=False).copy()
    dt = pd.Timestamp(sorted_df["rebalance_date"].iloc[0]).normalize()
    if policy == "v2_raw_no_refill":
        return sorted_df.head(stock_num)["code"].astype(str).tolist()

    selected = []
    selected_groups = {}
    for _, row in sorted_df.iterrows():
        code = str(row["code"])
        op = get_open_price(price_map, dt, code)
        if not can_buy_lot(op, target_value):
            continue
        if policy == "theme_cap1_refill":
            grp = row.get("etf_group", classify_etf_group(code, row.get("name", "")))
            if selected_groups.get(grp, 0) >= 1:
                continue
            selected_groups[grp] = selected_groups.get(grp, 0) + 1
        selected.append(code)
        if len(selected) >= stock_num:
            break
    return selected


def mark_to_market(cash, positions, price_map, date):
    total = float(cash)
    missing = []
    for code, shares in positions.items():
        op = get_open_price(price_map, date, code)
        if pd.isnull(op) or op <= 0:
            missing.append(code)
            continue
        total += float(shares) * float(op)
    return total, missing


def sell_position(code, shares, cash, price_map, date):
    op = get_open_price(price_map, date, code)
    if pd.isnull(op) or op <= 0:
        return cash, 0.0, False
    sell_price = max(float(op) - FIXED_SLIPPAGE, 0.0001)
    gross = float(shares) * sell_price
    commission = gross * COMMISSION_RATE
    return cash + gross - commission, gross, True


def buy_position(code, target_value, cash, price_map, date):
    op = get_open_price(price_map, date, code)
    if pd.isnull(op) or op <= 0:
        return cash, 0, 0.0, False, "no_price"
    buy_price = float(op) + FIXED_SLIPPAGE
    if target_value < buy_price * LOT_SIZE:
        return cash, 0, 0.0, False, "min_lot"
    shares = int(math.floor(target_value / buy_price / LOT_SIZE) * LOT_SIZE)
    if shares <= 0:
        return cash, 0, 0.0, False, "zero_lot"
    cost = shares * buy_price
    commission = cost * COMMISSION_RATE
    total_cost = cost + commission
    if total_cost > cash:
        shares = int(math.floor(cash / (buy_price * (1.0 + COMMISSION_RATE)) / LOT_SIZE) * LOT_SIZE)
        if shares <= 0:
            return cash, 0, 0.0, False, "cash"
        cost = shares * buy_price
        commission = cost * COMMISSION_RATE
        total_cost = cost + commission
    return cash - total_cost, shares, total_cost, True, "ok"


def simulate_one_model(score_data, model_name, stock_num, initial_cash, policy, price_map):
    sdf = score_data[score_data["model_name"] == model_name].copy()
    sdf = sdf.dropna(subset=["rebalance_date"])
    if sdf.empty:
        return pd.DataFrame(), pd.DataFrame()
    dates = sorted(pd.to_datetime(sdf["rebalance_date"].dropna().unique()))
    cash = float(initial_cash)
    positions = {}
    equity_rows = []
    target_rows = []
    last_equity = float(initial_cash)

    for dt in dates:
        gdf = sdf[sdf["rebalance_date"] == dt].copy()
        if gdf.empty:
            continue
        pre_value, missing = mark_to_market(cash, positions, price_map, dt)
        target_value_for_select = pre_value / float(max(1, stock_num))
        targets = select_target_codes(gdf, stock_num, policy, target_value_for_select, price_map)

        sold_value = 0.0
        for code in list(positions.keys()):
            if code not in targets:
                cash, gross, ok = sell_position(code, positions[code], cash, price_map, dt)
                if ok:
                    sold_value += gross
                    del positions[code]
        current_holds = [c for c in positions.keys() if positions.get(c, 0) > 0]
        buy_list = [c for c in targets if c not in current_holds]
        slots = len(targets) - len(current_holds)
        bought_value = 0.0
        buy_fail = []
        if slots > 0 and cash > 0:
            value = cash / float(slots)
            for code in buy_list:
                cash, shares, cost, ok, reason = buy_position(code, value, cash, price_map, dt)
                if ok:
                    positions[code] = positions.get(code, 0) + shares
                    bought_value += cost
                else:
                    buy_fail.append(code + ":" + reason)
        post_value, missing_post = mark_to_market(cash, positions, price_map, dt)
        # Performance from previous rebalance to current rebalance should be measured before submitting new trades.
        # post_value includes same-day trade costs and is used as the next period's starting equity.
        period_ret = pre_value / last_equity - 1.0 if last_equity > 0 else np.nan
        last_equity = post_value
        equity_rows.append({
            "model_name": model_name,
            "model_file": gdf["model_file"].iloc[0],
            "target_name": gdf["target_name"].iloc[0],
            "train_tag": gdf["train_tag"].iloc[0],
            "stock_num": int(stock_num),
            "initial_cash": float(initial_cash),
            "select_policy": policy,
            "date": pd.Timestamp(dt).normalize(),
            "equity": float(post_value),
            "cash": float(cash),
            "period_ret": float(period_ret),
            "target_count": int(len(targets)),
            "hold_count": int(len(positions)),
            "targets": ",".join(targets),
            "holds": ",".join(sorted(positions.keys())),
            "sold_value": float(sold_value),
            "bought_value": float(bought_value),
            "buy_fail": ",".join(buy_fail),
            "missing_price": ",".join(missing + missing_post),
        })
        rank_map = dict((str(row["code"]), int(i + 1)) for i, (_, row) in enumerate(gdf.sort_values("score", ascending=False).iterrows()))
        for code in targets:
            target_rows.append({
                "model_name": model_name,
                "stock_num": int(stock_num),
                "initial_cash": float(initial_cash),
                "select_policy": policy,
                "date": pd.Timestamp(dt).normalize(),
                "code": code,
                "rank": rank_map.get(code, np.nan),
                "open_price": get_open_price(price_map, dt, code),
            })
    return pd.DataFrame(equity_rows), pd.DataFrame(target_rows)


def summarize_equity(eq):
    if eq is None or eq.empty:
        return {}
    x = eq.sort_values("date").copy()
    if len(x) <= 1:
        return {"periods": int(len(x))}
    r_raw = pd.to_numeric(x["period_ret"], errors="coerce")
    missing_mask = x["missing_price"].fillna("").astype(str).str.len() > 0
    extreme_mask = r_raw.abs() > 1.0
    invalid_mask = missing_mask | extreme_mask | r_raw.isnull()
    buy_fail_mask = x["buy_fail"].fillna("").astype(str).str.len() > 0
    if invalid_mask.any():
        return {
            "periods": int(len(x)),
            "total_ret": np.nan,
            "mean_period_ret": np.nan,
            "win_rate": np.nan,
            "max_drawdown": np.nan,
            "weekly_sharpe": np.nan,
            "avg_hold_count": float(pd.to_numeric(x["hold_count"], errors="coerce").mean()),
            "buy_fail_weeks": int(buy_fail_mask.sum()),
            "invalid_price_weeks": int(missing_mask.sum()),
            "extreme_ret_weeks": int(extreme_mask.sum()),
            "valid_summary": False,
        }
    r = r_raw.fillna(0.0)
    equity = pd.to_numeric(x["equity"], errors="coerce")
    dd = equity / equity.cummax() - 1.0
    total_ret = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] > 0 else np.nan
    std = r.std()
    return {
        "periods": int(len(x)),
        "total_ret": total_ret,
        "mean_period_ret": float(r.mean()),
        "win_rate": float((r > 0).mean()),
        "max_drawdown": float(dd.min()),
        "weekly_sharpe": float(r.mean() / std) if (not pd.isnull(std)) and std > 0 else np.nan,
        "avg_hold_count": float(pd.to_numeric(x["hold_count"], errors="coerce").mean()),
        "buy_fail_weeks": int(buy_fail_mask.sum()),
        "invalid_price_weeks": int(missing_mask.sum()),
        "extreme_ret_weeks": int(extreme_mask.sum()),
        "valid_summary": True,
    }

if RUN_JQ_LIKE_SIM and (open_price_df is not None) and (not open_price_df.empty) and (score_df is not None) and (not score_df.empty):
    price_map = make_price_map(open_price_df)
    equity_parts = []
    target_parts = []
    model_names = sorted(score_df["model_name"].dropna().unique().tolist())
    for model_name in tqdm(model_names, desc="jq-like models"):
        for stock_num in STOCK_NUM_LIST:
            for initial_cash in INITIAL_CASH_LIST:
                for policy in SELECT_POLICIES:
                    eq, tg = simulate_one_model(score_df, model_name, stock_num, initial_cash, policy, price_map)
                    if not eq.empty:
                        equity_parts.append(eq)
                    if not tg.empty:
                        target_parts.append(tg)
                    gc.collect()
    jq_equity_df = pd.concat(equity_parts, ignore_index=True, sort=False) if equity_parts else pd.DataFrame()
    jq_targets_df = pd.concat(target_parts, ignore_index=True, sort=False) if target_parts else pd.DataFrame()
else:
    jq_equity_df = pd.DataFrame()
    jq_targets_df = pd.DataFrame()

summary_rows = []
if not jq_equity_df.empty:
    group_cols = ["model_name", "stock_num", "initial_cash", "select_policy"]
    for keys, gdf in jq_equity_df.groupby(group_cols):
        row = dict(zip(group_cols, keys))
        row.update(summarize_equity(gdf))
        first = gdf.iloc[0]
        row.update({"model_file": first.get("model_file"), "target_name": first.get("target_name"), "train_tag": first.get("train_tag")})
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["total_ret", "max_drawdown"], ascending=[False, False])

jq_equity_df.to_csv(JQ_LIKE_EQUITY_CSV, index=False)
jq_targets_df.to_csv(JQ_LIKE_TARGETS_CSV, index=False)
summary_df.to_csv(JQ_LIKE_SUMMARY_CSV, index=False)
print("jq equity:", jq_equity_df.shape, "targets:", jq_targets_df.shape, "summary:", summary_df.shape)
display(summary_df.head(40))

In [ ]:
# =========================
# 7. Yearly breakdown and latest targets
# =========================
def build_yearly_from_equity(eq):
    rows = []
    if eq is None or eq.empty:
        return pd.DataFrame()
    group_cols = ["model_name", "stock_num", "initial_cash", "select_policy"]
    tmp = eq.copy()
    tmp["date"] = pd.to_datetime(tmp["date"])
    tmp["year"] = tmp["date"].dt.year
    for keys, gdf0 in tmp.groupby(group_cols):
        key_row = dict(zip(group_cols, keys))
        for year, gdf in gdf0.groupby("year"):
            gdf = gdf.sort_values("date").copy()
            equity = pd.to_numeric(gdf["equity"], errors="coerce")
            if len(equity) == 0 or equity.iloc[0] <= 0:
                continue
            r_raw = pd.to_numeric(gdf["period_ret"], errors="coerce")
            missing_mask = gdf["missing_price"].fillna("").astype(str).str.len() > 0
            extreme_mask = r_raw.abs() > 1.0
            invalid_mask = missing_mask | extreme_mask | r_raw.isnull()
            buy_fail_mask = gdf["buy_fail"].fillna("").astype(str).str.len() > 0
            if invalid_mask.any():
                dd_min = np.nan
                year_ret = np.nan
                win_rate = np.nan
            else:
                dd = equity / equity.cummax() - 1.0
                r = r_raw.fillna(0.0)
                dd_min = float(dd.min())
                year_ret = float(equity.iloc[-1] / equity.iloc[0] - 1.0)
                win_rate = float((r > 0).mean())
            row = dict(key_row)
            row.update({
                "year": int(year),
                "periods": int(len(gdf)),
                "year_ret": year_ret,
                "max_drawdown": dd_min,
                "win_rate": win_rate,
                "avg_hold_count": float(pd.to_numeric(gdf["hold_count"], errors="coerce").mean()),
                "buy_fail_weeks": int(buy_fail_mask.sum()),
                "invalid_price_weeks": int(missing_mask.sum()),
                "extreme_ret_weeks": int(extreme_mask.sum()),
                "valid_summary": bool(not invalid_mask.any()),
            })
            first = gdf.iloc[0]
            row.update({"model_file": first.get("model_file"), "target_name": first.get("target_name"), "train_tag": first.get("train_tag")})
            rows.append(row)
    return pd.DataFrame(rows)


def build_latest_targets(score_data, stock_num=3, policy=MAIN_SELECT_POLICY):
    rows = []
    if score_data is None or score_data.empty:
        return pd.DataFrame()
    for model_name, gdf0 in score_data.groupby("model_name"):
        latest_date = pd.to_datetime(gdf0["feature_date"]).max()
        gdf = gdf0[gdf0["feature_date"] == latest_date].copy()
        gdf = gdf.sort_values("score", ascending=False)
        top = select_rows_by_policy(gdf, stock_num, "theme_cap1_refill" if policy == "theme_cap1_refill" else "v2_raw_no_refill")
        for rank_idx, (_, row) in enumerate(top.iterrows(), 1):
            rows.append({
                "model_name": model_name,
                "model_file": row.get("model_file"),
                "target_name": row.get("target_name"),
                "train_tag": row.get("train_tag"),
                "feature_date": latest_date,
                "rebalance_date": row.get("rebalance_date"),
                "rank": int(rank_idx),
                "code": row.get("code"),
                "name": row.get("name", ""),
                "etf_group": row.get("etf_group", ""),
                "score": row.get("score"),
            })
    return pd.DataFrame(rows)

yearly_df = build_yearly_from_equity(jq_equity_df)
if not yearly_df.empty:
    yearly_df = yearly_df.sort_values(["model_name", "stock_num", "initial_cash", "select_policy", "year"])
yearly_df.to_csv(JQ_LIKE_YEARLY_CSV, index=False)
latest_targets_df = build_latest_targets(score_df, stock_num=3, policy=MAIN_SELECT_POLICY)
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)

print("yearly:", yearly_df.shape)
display(yearly_df.head(80))
print("latest targets:")
display(latest_targets_df.head(40))

## 8. 结果解读顺序

跑完后按这个顺序看，不要直接看最高累计收益：

1. `etf_ml_v8_jq_like_summary.csv`：先筛选 `select_policy == v2_raw_no_refill`，看 `ret5d + train20210101_20251231` 是否仍然领先。
2. `etf_ml_v8_jq_like_yearly.csv`：看 2024、2025、2026 是否都能接受，尤其 2026 是否只是靠单月行情。
3. 对比 `stock_num=1` 和 `stock_num=3`：如果 top1 明显更强，说明模型头部排序更有价值；如果 top3 更稳，说明分散有意义。
4. 对比 `initial_cash`：如果小资金结果差很多，说明真实执行受高价 ETF/100 股约束影响大。
5. 对比 `theme_cap1_refill`：如果主题 cap 收益大幅下降但回撤没改善，就继续放弃主题限制。
6. `close_proxy` 只用于解释信号，不用于最终选模型。

保留标准：JQ-like 和真实回测方向一致，且分年结果不是只靠单一年度/单一主题。